In [14]:
%matplotlib inline

import sys
import datacube
import skimage.exposure
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from IPython.core.display import Video
import xarray as xr
import numpy as np
import os
import imageio
import pandas as pd
from PIL import Image
from pyproj import Transformer

from matplotlib import colors as mcolours

sys.path.insert(1, "../Tools/")
from dea_tools.bandindices import calculate_indices
from dea_tools.plotting import xr_animation, rgb, display_map
from dea_tools.landcover import get_layer_name, lc_colours, lc_colourmap, make_colorbar

# Connect to dev sandbox database

In [15]:
dc = datacube.Datacube(app="lc3_vs_lc2")  # for this to work, need to paste the command at https://docs.dev.dea.ga.gov.au/procedures/dev_database_from_prod_jupyterhub.html in the terminal

# Input data and folder paths
Add things here that are needed to run the rest of the notebook. This way the rest of the notebook can be left alone and shouldn't need to be edited.


In [16]:
df_ROIs = pd.read_csv('input_query.csv')
df_ROIs.columns # check the column names of the dataframe

Index(['centre_x', 'centre_y', 'buffer_size_wgs84', 'start_year', 'end_year',
       'interval', 'name', 'level', 'level3_class', 'level4_class', 'comment'],
      dtype='object')

In [17]:
df_ROIs

,centre_x,centre_y,buffer_size_wgs84,start_year,end_year,interval,name,level,level3_class,level4_class,comment
0,150.509,-30.83261,0.15,2010,2020,400,NSW_lake_keepit_change,4,NaN,"101, 102",changing water levels in lake keepit over time
1,115.700,-31.66800,0.05,2010,2020,400,WA_urban_expansion,4,NaN,93,urban expansion


In [18]:
# select a level/measurement of interest 
var_name = 'level4'
classes_list = [93, 95]

In [19]:
# define the variables to store data needed for the query in dc.load()
# the following coords ranges are created from centroids + buffer size 
x_min = df_ROIs['centre_x'] - df_ROIs['buffer_size_wgs84']/2
x_max = df_ROIs['centre_x'] + df_ROIs['buffer_size_wgs84']/2
y_min = df_ROIs['centre_y'] - df_ROIs['buffer_size_wgs84']/2
y_max = df_ROIs['centre_y'] + df_ROIs['buffer_size_wgs84']/2
time_start = df_ROIs['start_year']
time_end = df_ROIs['end_year']
ROI_names = df_ROIs['name']


In [20]:
# define resolution of output in metres. For LC V2 it's 30
output_res = 30

In [21]:
# define LC product of interest 
lc_product = 'ga_ls_landcover_class_cyear_3'

# Connect to dev sandbox database

In [22]:
output_dir = 'count_pixels_area' # define folder where to save output

# create output folders if they do not exist already
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Useful functions

In [23]:
def get_query(lat_range, 
            lon_range,  
            year):
    '''
    Generates dc.load() query
    -------------------------
    lat_range, lon_range : tuple
        coordinates ranges, i.e. ROI
    year: int for individual years; tuple of 2 elements, or list of 2 elements for a range of years
        year or years of interest
    '''

    try: # if year is provide as an integer, this will fail because cannot use len(), and will go to the except
        if len(year) == 2: # if tuple or list containing 2 elements, the query will be for a time range
            query = {
                    "y": lat_range,
                    "x": lon_range,
                    "time": [str(y) for y in year],
                    }
        else:
            raise KeyError('`year` must be either an integer, a tuple of 2 elements, or a list of 2 elements')

    except: # if integer, select individual year
        query = {
                "y": lat_range,
                "x": lon_range,
                "time": str(year),
                }

    return query

In [24]:
# functions for creating output df (one for each ROI, because each df will have mulitple rows representing the years in the timeseries)
def create_empty_out_df(columns):
    df = pd.DataFrame(columns=columns)
    return df

In [25]:
def extract_data_one_ROI(index_input_df, 
                         input_df,  
                         ROI_names,
                         x_min, 
                         x_max, 
                         y_min, 
                         y_max,
                         time_start,
                         time_end,
                         lc_product,
                         var_name,
                         output_res,
                         classes_list = 'all'):
    '''
    Extract all data using input parameters taken from ROIs table
    -------------------------
    index_input_df : int
        index of input df where stored ROIs' information
    input_df : pandas dataframe
        dataframe with information on ROIs
    ROI_names : pandas series
        series containing names (IDs) or ROIs
    x_min, x_max, y_min, y_max : pandas series
        series containing coordinates of edges of ROIs
    time_start, time_end : pandas series
        series containing start and end years of timeseries of interest for all ROIs
    lc_product : str 
        name of LC product collection
    var_name : str 
        name of level of LC or variable name (e.g. "level3")
    output_res : int 
        resolution of output in metres
    classes_list : list or 'all' 
        either a list of values of LC classes of interest, or 'all' if count evry class present in ROIs
    '''

    #create output df 
    out_columns = ['centre_x', 'centre_y', 'buffer_size_wgs84', 'year', 'product_name', 'var_name', 'class_value', 'pixels_count', 'area_m2', 'area_km2', 'ha']
    
    out_df = create_empty_out_df(out_columns) 
    out_df_row = 0 # starting row to use for populating output df

    x = (x_min[index_input_df], x_max[index_input_df]) # lon range of ROI
    y = (y_min[index_input_df], y_max[index_input_df]) # lat range of ROI
    time = (time_start[index_input_df], time_end[index_input_df]) # time range
    
    print(i,x,y,time) # print to help matching plots with df rows
    
    # define query
    query = get_query(lat_range = y,
                 lon_range = x ,
                 year = time
                )

    # load LC collection 2
    lc_ds = dc.load(
        product=lc_product,
        output_crs='EPSG:3577', #GDA94
        measurements=[var_name],
        resolution=(-1*output_res, output_res), 
        **query
    )
    print('aaaa')
    # if selected 'all', modify list to contain all the classes present within the ROI 
    if classes_list == 'all':
        classes_list = np.unique(lc_ds[var_name].values)
    
    # count pixels of classes of interest for every year 
    for class_value in classes_list: 
        # iterate over years
        for t in lc_ds.time:
            lc_df_y = lc_ds.sel(time=t)
            
            px_count = np.nansum(lc_df_y[var_name] == class_value)
            metre_sq = px_count * (output_res**2)
            km_sq = metre_sq / 1000000
            ha = metre_sq / 10000
    
            # populate df
            print(f'\nadding data to row of index: {out_df_row}\n')
            
            out_df.loc[out_df_row,'centre_x'] = input_df['centre_x'][index_input_df]
            out_df.loc[out_df_row,'centre_y'] = input_df['centre_y'][index_input_df]
            out_df.loc[out_df_row,'buffer_size_wgs84'] = input_df['buffer_size_wgs84'][index_input_df]
            out_df.loc[out_df_row,'year'] = pd.to_datetime(t.values).year #convert time value into datetime and then extract the year
            out_df.loc[out_df_row,'product_name'] = lc_product
            out_df.loc[out_df_row,'var_name'] = var_name
            out_df.loc[out_df_row,'class_value'] = class_value
            out_df.loc[out_df_row,'pixels_count'] = px_count
            out_df.loc[out_df_row,'area_m2'] = metre_sq
            out_df.loc[out_df_row,'area_km2'] = km_sq
            out_df.loc[out_df_row,'ha'] = ha

            # increase output df row where to save output (i.e. got to next empty row)
            out_df_row += 1

    #save output df as csv 
    print('saving output...')
    out_df.to_csv(f'{output_dir}/count_pixels_and_area_{ROI_names[index_input_df]}.csv', index=False)
    print('output CSV file was saved')

    return

# Count pixels and area and save output as CSV

In [26]:
for i in df_ROIs.index: # iterate over ROIs

    var_name = f"level{df_ROIs['level'][i]}" # the LC level

    try: 
        classes_list = [int(i) for i in df_ROIs[f'{var_name}_class'][i].split(',')] # convert table cell to list of integers
    except: #if it fails, it's either because input is 'all' or there are typos ect 
        if df_ROIs[f'{var_name}_class'][i] == 'all':
            classes_list = 'all' 
        else:
            raise ValueError(f'Classes in row of index {i} are not provided in the correct format')
    
    extract_data_one_ROI( i, 
                        df_ROIs, 
                         ROI_names,
                         x_min, 
                         x_max, 
                         y_min, 
                         y_max,
                         time_start,
                         time_end,
                         lc_product,
                         var_name,
                         output_res,
                         classes_list)
                            

0 (150.434, 150.58399999999997) (-30.90761, -30.75761) (2010, 2020)


ProgrammingError: (psycopg2.errors.InsufficientPrivilege) permission denied for schema agdc
LINE 2: FROM agdc.dataset_type ORDER BY agdc.dataset_type.name ASC
             ^

[SQL: SELECT agdc.dataset_type.id, agdc.dataset_type.name, agdc.dataset_type.metadata, agdc.dataset_type.metadata_type_ref, agdc.dataset_type.definition, agdc.dataset_type.added, agdc.dataset_type.added_by 
FROM agdc.dataset_type ORDER BY agdc.dataset_type.name ASC]
(Background on this error at: https://sqlalche.me/e/14/f405)